In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

In [4]:
## Step 1 — generate random forcing functions
np.random.seed(0)
def random_u(t, n_terms=5):
    """Generate one random smooth forcing function evaluated at times t."""
    u = np.zeros_like(t)
    for _ in range(n_terms):
        a = np.random.uniform(-1, 1)
        w = np.random.uniform(1, 5)      # frequency
        phi = np.random.uniform(0, 2*np.pi)
        u += a * np.sin(w * t + phi)
    return u

In [5]:
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

def solve_v(t_dense, u_dense, t_eval):
    u_interp = interp1d(t_dense, u_dense, fill_value="extrapolate")
    def rhs(t, v):
        return -v + u_interp(t)
    sol = solve_ivp(rhs, [0, 1], [0.0], t_eval=t_eval, dense_output=True)
    return sol.y[0]

In [41]:
m = 20                                  # number of branch sensors
x_sensors = np.linspace(0, 1, m)        # fixed sensor locations for u
t_dense = np.linspace(0, 1, 200)        # fine grid to integrate the ODE accurately

n_samples = 500
n_query = 15                            # query points per sample (can vary!)

U_branch = []   # shape (n_samples, m)      -> branch input
Y_query  = []   # shape (n_samples, n_query) -> trunk input
V_target = []   # shape (n_samples, n_query) -> ground truth output

for i in range(n_samples):
    u_dense = random_u(t_dense)
    u_sensor_vals = random_u(x_sensors)  # NOTE: same random draw, see below

    y_q = np.sort(np.random.uniform(0, 1, n_query))
    v_true = solve_v(t_dense, u_dense, y_q)

    U_branch.append(u_sensor_vals)
    Y_query.append(y_q)
    V_target.append(v_true)

In [ ]:
class DeepONet(nn.Module):
    def __init__(self, m, hidden_layer):
        super().__init__()
        self.fcb1 = nn.Linear(m, 100)
        self.fcb2 = nn.Linear(100, hidden_layer)
        self.fct1 = nn.Linear(1, 100)
        self.fct2 = nn.Linear(100, hidden_layer)
        self.b0 = nn.Parameter(torch.zeros(1))

    def forward(self, xb, xt):
        xb = nn.functional.relu(self.fcb1(xb))
        xb = nn.functional.relu(self.fcb2(xb))          # (n_samples, hidden_layer)

        n_query = xt.shape[0] // xb.shape[0]
        xb_repeated = xb.repeat_interleave(n_query, dim=0)  # (n_samples*n_query, hidden_layer)

        xt = nn.functional.relu(self.fct1(xt))
        xt = nn.functional.relu(self.fct2(xt))            # (n_samples*n_query, hidden_layer)

        x_combined = torch.sum(xb_repeated * xt, dim=1, keepdim=True) + self.b0
        return x_combined   # (n_samples*n_query, 1)      

In [43]:
model	= DeepONet(m, hidden_layer=100)

epochs = 1000
loss_fn	= nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_history = []
xb_tensor = torch.tensor(U_branch, dtype=torch.float32)
xt_tensor = torch.tensor(Y_query, dtype=torch.float32).reshape(-1, 1)
v_target_tensor = torch.tensor(V_target, dtype=torch.float32).reshape(-1, 1)

for epoch in range(epochs):
    v_pred	= model(xb_tensor, xt_tensor)
    loss = loss_fn(v_pred, v_target_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")

Epoch 1/1000, Loss: 1.677004
Epoch 2/1000, Loss: 0.954615
Epoch 3/1000, Loss: 0.554307
Epoch 4/1000, Loss: 0.339258
Epoch 5/1000, Loss: 0.227897
Epoch 6/1000, Loss: 0.170465
Epoch 7/1000, Loss: 0.141197
Epoch 8/1000, Loss: 0.125835
Epoch 9/1000, Loss: 0.117692
Epoch 10/1000, Loss: 0.113473
Epoch 11/1000, Loss: 0.111337
Epoch 12/1000, Loss: 0.110121
Epoch 13/1000, Loss: 0.109422
Epoch 14/1000, Loss: 0.108952
Epoch 15/1000, Loss: 0.108646
Epoch 16/1000, Loss: 0.108455
Epoch 17/1000, Loss: 0.108331
Epoch 18/1000, Loss: 0.108249
Epoch 19/1000, Loss: 0.108187
Epoch 20/1000, Loss: 0.108145
Epoch 21/1000, Loss: 0.108112
Epoch 22/1000, Loss: 0.108088
Epoch 23/1000, Loss: 0.108067
Epoch 24/1000, Loss: 0.108049
Epoch 25/1000, Loss: 0.108034
Epoch 26/1000, Loss: 0.108022
Epoch 27/1000, Loss: 0.108014
Epoch 28/1000, Loss: 0.108005
Epoch 29/1000, Loss: 0.107998
Epoch 30/1000, Loss: 0.107993
Epoch 31/1000, Loss: 0.107988
Epoch 32/1000, Loss: 0.107985
Epoch 33/1000, Loss: 0.107982
Epoch 34/1000, Loss